# Reconciling Campaign Runs onto the Calendar

This notebook demonstrates `solsys_code/campaign_reconciler.py` and
`solsys_code/management/commands/reconcile_campaign_runs.py` (Phase 29, "the reconciler"),
the single idempotent function/command pair that projects and refreshes `CalendarEvent`
rows for every `CampaignRun`, replacing the retired per-gap range-window backfill
command.

It demonstrates:

- Seeding two `Observatory` rows (a ground site with a real IANA timezone, and a satellite
  site) and a campaign `TargetList`
- Seeding four `CampaignRun` rows that exercise the reconciler's three dispatch branches:
  two per-night runs (a classically-scheduled run and a queue-scheduled run, both at the
  same kind of resolved ground site) and two whole-window container runs (a class-wide
  allocation and a satellite run)
- Why a run's `source` field is still worth setting correctly for provenance and reporting,
  even though it no longer decides which calendar-event family a run gets -- only
  `telescope_class`/`site` do
- A `--dry-run` sweep via `call_command`, showing the `would_create` counters and that no
  `CalendarEvent` rows are written
- A real sweep, then a printed loop over the resulting events showing the two coexisting
  key families: date-bearing `RUN:{pk}:{date}` rows for the classical and queue-scheduled
  runs, and a single bare `RUN:{pk}` row each for the class-wide and satellite runs
- A second real sweep reporting `created: 0, updated: 0` -- the idempotency claim
  demonstrated, not just asserted

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory`/`TargetList` records and creates `CampaignRun`/`CalendarEvent` rows) and is
therefore **NOT** run during Sphinx/CI/ReadTheDocs builds, per `docs/notebooks/README.md`.

## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` -- three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.

Before `django.setup()` runs, the setup cell below also copies the developer database (`src/fomo_db.sqlite3`) to a throwaway scratch file and points `FOMO_DATABASE_PATH` at that copy, so this notebook's entire run -- every row it creates or removes -- happens against a copy that is torn down at the end, never against the developer database itself (UAT G-33-4).


In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
if not (repo_root_path / 'manage.py').exists():
    raise RuntimeError(f'No manage.py at {repo_root_path}; run Jupyter from docs/notebooks/pre_executed/')
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

# Copy the developer database to a scratch file and point FOMO_DATABASE_PATH at the
# copy BEFORE django.setup() -- that call is what materialises DATABASES, so the
# variable must already be exported. This is what makes the developer database
# read-only for this notebook's entire run (UAT G-33-4).
import shutil
import tempfile

dev_db_path = repo_root_path / 'src' / 'fomo_db.sqlite3'
if not dev_db_path.exists():
    raise RuntimeError(f'No developer database at {dev_db_path}; run `python manage.py migrate` first.')
scratch_db_dir = Path(tempfile.mkdtemp(prefix='fomo-notebook-db-'))
scratch_db_path = scratch_db_dir / 'fomo_db.sqlite3'
shutil.copy2(dev_db_path, scratch_db_path)
os.environ['FOMO_DATABASE_PATH'] = str(scratch_db_path)

django.setup()

# This notebook intentionally imports only the campaign-coordination models and the
# reconciler below -- never the ephemeris view/computation modules, which trigger a large
# one-time SPICE kernel download on first import.

from django.conf import settings as django_settings

resolved_db_name = django_settings.DATABASES['default']['NAME']
assert resolved_db_name == str(scratch_db_path), (
    'This notebook must never write to the developer database -- resolved DB '
    f'{resolved_db_name!r} is not the scratch copy {str(scratch_db_path)!r}.'
)

print(f'Django ready: settings module={os.environ["DJANGO_SETTINGS_MODULE"]!r}, repo_root={repo_root!r}')
print(f'Resolved database: {resolved_db_name!r} (scratch copy of the developer database)')

Django ready: settings module='src.fomo.settings', repo_root='/home/tlister/git/fomo_devel'
Resolved database: '/tmp/fomo-notebook-db-4d2o_ad7/fomo_db.sqlite3' (scratch copy of the developer database)


## Seed Observatory records and the campaign TargetList

The reconciler needs one resolvable ground `Observatory` (with a real IANA `timezone`, so
`sun_event()` can compute dip-corrected sunset/sunrise) and one satellite `Observatory`
(`observations_type=SATELLITE_OBSTYPE`, no fixed horizon -- the reconciler's container branch
skips the per-night sun math for these entirely). `update_or_create` makes this cell
idempotent -- safe to re-run against any dev DB.

The campaign container is a `tom_targets.models.TargetList`, found-or-created by name.

In [2]:
from tom_targets.models import TargetList

from solsys_code.solsys_code_observatory.models import Observatory

ground_site, _ = Observatory.objects.update_or_create(
    obscode='X29',
    defaults=dict(
        name='Reconciler Demo Ground Site',
        short_name='RDGS',
        lat=-29.2567,
        lon=-70.7300,
        altitude=2347,
        timezone='America/Santiago',
        observations_type=Observatory.OPTICAL_OBSTYPE,
    ),
)
print(f'Ground site:    obscode={ground_site.obscode!r}  timezone={ground_site.timezone!r}')

satellite_site, _ = Observatory.objects.update_or_create(
    obscode='X30',
    defaults=dict(
        name='Reconciler Demo Space Telescope',
        short_name='RDST',
        observations_type=Observatory.SATELLITE_OBSTYPE,
    ),
)
print(f'Satellite site: obscode={satellite_site.obscode!r}  observations_type=SATELLITE_OBSTYPE')

campaign, campaign_created = TargetList.objects.get_or_create(name='Reconciler Demo Campaign')
print(f'\nCampaign: {campaign.name!r} (pk={campaign.pk}) {"created" if campaign_created else "found"}')

Ground site:    obscode='X29'  timezone='America/Santiago'
Satellite site: obscode='X30'  observations_type=SATELLITE_OBSTYPE

Campaign: 'Reconciler Demo Campaign' (pk=9) found


## Seed four CampaignRun rows -- the three dispatch branches

`reconcile_run()` dispatches on the run's own state, in this order: a non-blank
`telescope_class` (class-wide/space) always wins first, then a satellite `site`, and
only then the classical per-night branch -- which every other approved, windowed run
with a resolved ground site takes, whether it is queue-scheduled or not. The four rows
below exercise all three branches (two of them share the per-night branch):

1. **Classical** -- a non-queue `source`, a resolved ground site, and a 3-night window.
   Projects one `CalendarEvent` per observing night.
2. **Queue** -- `source` set to `CampaignRun.Source.LCO_QUEUE`, but with the SAME kind of
   resolved ground site as the classical run. Projects one `CalendarEvent` per observing
   night too -- queue-sourcing alone does not change the shape.
3. **Satellite** -- a `site` with `observations_type=SATELLITE_OBSTYPE` (no fixed horizon).
   Projects a single bare whole-window container event.
4. **Class-wide** -- `telescope_class` set, no site at all. Also projects a single bare
   whole-window container event.

**Why `source` is still set explicitly here, and what it means for real data:** the
reconciler never branches on `CampaignRun.source` (never a text heuristic over
`telescope_instrument`/`site_raw` either) -- only a non-blank `telescope_class` or a
satellite `site` select the whole-window container branch; every other approved, windowed
run with a resolved site takes the per-night branch, queue-scheduled or not. `source` is
still worth setting correctly, though: it is the run's provenance record (where it came
from), and staff-facing views and reports read it, even though it no longer changes an
event's window shape. See "How do I get every campaign run onto the calendar?" in
`docs/runbooks/telescope_runs_calendar.rst` for the operator-facing version of this note.
**What happens if a run's `telescope_class` or `site` changes after it was already
reconciled once under its old classification:** `reconcile_run()` re-derives which
calendar-event family a run belongs to from its current `telescope_class`/`site` on every
call, so a later reconcile automatically detaches (never deletes) the old family's
events -- returning them to the attribution page's worklist for a staff member to
re-confirm or discard -- rather than leaving them on the calendar looking like a live
commitment forever. See `campaign_reconciler._detach_stale_family_events()` and "Can I
correct a run's source?" in `docs/runbooks/telescope_runs_calendar.rst` for the full
explanation. Since Phase 33 (D-16), that same detach also clears the row's
`confirmed_by`/`confirmed_at` audit stamps together with the link -- a detached row
never goes on displaying a confirmation for an attribution that no longer exists.

In [3]:
from datetime import date

from solsys_code.models import CampaignRun

classical_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDGS/EFOSC2',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 3),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='Classical multi-night photometric monitoring (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.CLASSICAL_FILE,
    ),
)
print(
    f'Classical run:  pk={classical_run.pk}  source={classical_run.source!r}  '
    f'window={classical_run.window_start}..{classical_run.window_end}'
)

queue_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDGS 1m0-SciCam-Sinistro',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 3),
    defaults=dict(
        site=ground_site,
        site_raw='X29',
        observation_details='LCO queue allocation, resolved ground site (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LCO_QUEUE,
    ),
)
print(
    f'Queue run:      pk={queue_run.pk}  source={queue_run.source!r}  '
    f'window={queue_run.window_start}..{queue_run.window_end}  '
    f'(resolved site -- per-night branch, same as the classical run)'
)

satellite_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='RDST Space Telescope',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 5),
    defaults=dict(
        site=satellite_site,
        site_raw='X30',
        observation_details='Satellite allocation, no fixed horizon (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LEGACY,
    ),
)
print(
    f'Satellite run:  pk={satellite_run.pk}  site={satellite_run.site!r}  '
    f'window={satellite_run.window_start}..{satellite_run.window_end}'
)

class_wide_run, _ = CampaignRun.objects.update_or_create(
    campaign=campaign,
    telescope_instrument='LCO 1m0 Network (demo)',
    window_start=date(2026, 9, 1),
    window_end=date(2026, 9, 30),
    defaults=dict(
        site=None,
        telescope_class=CampaignRun.TelescopeClass.ONE_M0,
        observation_details='Class-wide 1m0 network allocation (demo)',
        approval_status=CampaignRun.ApprovalStatus.APPROVED,
        source=CampaignRun.Source.LEGACY,
    ),
)
print(
    f'Class-wide run: pk={class_wide_run.pk}  telescope_class={class_wide_run.telescope_class!r}  '
    f'site={class_wide_run.site!r}  window={class_wide_run.window_start}..{class_wide_run.window_end}'
)

Classical run:  pk=59  source=CampaignRun.Source.CLASSICAL_FILE  window=2026-09-01..2026-09-03
Queue run:      pk=60  source=CampaignRun.Source.LCO_QUEUE  window=2026-09-01..2026-09-03  (resolved site -- per-night branch, same as the classical run)
Satellite run:  pk=61  site=<Observatory: X30: Reconciler Demo Space Telescope>  window=2026-09-01..2026-09-05
Class-wide run: pk=62  telescope_class=CampaignRun.TelescopeClass.ONE_M0  site=None  window=2026-09-01..2026-09-30


## Dry run first

`--dry-run` reports the `would_create`/`would_update`/`would_leave_unchanged` counters
without writing a single `CalendarEvent` row -- always run this before a real sweep.

In [4]:
import io

from django.core.management import call_command
from tom_calendar.models import CalendarEvent

events_before_dry_run = CalendarEvent.objects.count()

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()
call_command('reconcile_campaign_runs', '--dry-run', stdout=stdout_buf, stderr=stderr_buf)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr:', stderr_buf.getvalue())

events_after_dry_run = CalendarEvent.objects.count()
print(f'CalendarEvent.objects.count() before dry run: {events_before_dry_run}')
print(f'CalendarEvent.objects.count() after dry run:  {events_after_dry_run}')
assert events_after_dry_run == events_before_dry_run, 'A --dry-run sweep must never write a CalendarEvent row'

stdout: Done (dry run). runs: 54, would_create: 0, would_update: 0, would_leave_unchanged: 82, skipped: 10, failed: 0, blocked: 0, skipped_nights: 0, would_detach: n/a (dry-run)

stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)
Run pk=73: skipped (not approved)

CalendarEvent.objects.count() before dry run: 257
CalendarEvent.objects.count() after dry run:  257


## The real sweep

Running the same sweep without `--dry-run` now writes the calendar events. The printed
loop below inspects only the four runs seeded above (via `campaign_reconciler.owned_events()`,
the same ownership-scoped query the reconciler itself uses), making the two coexisting key
families directly visible: the classical and queue-scheduled runs each get one date-bearing
`RUN:{pk}:{date}` event per night (queue-sourcing alone does not change the shape -- both
have a resolved ground site), while the class-wide and satellite runs each get a single bare
`RUN:{pk}` container event spanning their whole window.

Submitters write a run's Telescope / Instrument as free text using `/` or `+` (for example `'RDGS/EFOSC2'`). The reconciler splits that text on the first delimiter into the calendar event's two separate `telescope` and `instrument` fields -- exactly what the event-detail pop-up renders as its "Telescope" and "Instrument" boxes. A value with no delimiter at all, like the satellite and class-wide runs' `telescope_instrument` below, goes wholly into `telescope` with `instrument` left blank.


In [5]:
from solsys_code.campaign_reconciler import owned_events

stdout_buf_real = io.StringIO()
stderr_buf_real = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_real, stderr=stderr_buf_real)

print('stdout:', stdout_buf_real.getvalue())
if stderr_buf_real.getvalue():
    print('stderr:', stderr_buf_real.getvalue())

for label, run in [
    ('Classical', classical_run),
    ('Queue', queue_run),
    ('Satellite', satellite_run),
    ('Class-wide', class_wide_run),
]:
    print(f'--- {label} run (pk={run.pk}) ---')
    for ev in owned_events(run).order_by('start_time'):
        print(f'  url={ev.url!r}')
        print(f'    title={ev.title!r}')
        print(f'    telescope={ev.telescope!r}  instrument={ev.instrument!r}')
        print(f'    start={ev.start_time.isoformat()}  end={ev.end_time.isoformat()}')
    print()

stdout: Done. runs: 54, created: 0, updated: 0, unchanged: 82, skipped: 10, failed: 0, blocked: 0, skipped_nights: 0, detached: 0

stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)
Run pk=73: skipped (not approved)

--- Classical run (pk=59) ---
  url='RUN:59:2026-09-01'
    title='RDGS/EFOSC2 (window 2026-09-01..2026-09-03)'
    telescope='RDGS'  instrument='EFOSC2'
    start=2026-09-01T22:35:09+00:00  end=2026-09-02T10:49:50+00:00
  url='RUN:59:2026-09-02'
    title='RDGS/EFOSC2 (window 2026-09-01..2026-09-03)'
    telescope='RDGS'  instrument='EFOSC2'
    start=2026-09-02T22:35:39+00:00  end=2026-09-03T10:48:41+00:00
  url='RUN:59:2026-09-03'
    title='RDGS/EFOSC2 (window 2026-09-01..2026-09-03)'
    telescope='RDGS'  instrument='

## Idempotency: a second sweep changes nothing

Running `reconcile_campaign_runs` again against unchanged run state must report
`created: 0, updated: 0` -- demonstrated below rather than asserted, per RECON-01.

In [6]:
stdout_buf_second = io.StringIO()
stderr_buf_second = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_second, stderr=stderr_buf_second)

print('Second sweep stdout:', stdout_buf_second.getvalue())
if stderr_buf_second.getvalue():
    print('Second sweep stderr:', stderr_buf_second.getvalue())

assert 'created: 0' in stdout_buf_second.getvalue()
assert 'updated: 0' in stdout_buf_second.getvalue()

Second sweep stdout: Done. runs: 54, created: 0, updated: 0, unchanged: 82, skipped: 10, failed: 0, blocked: 0, skipped_nights: 0, detached: 0

Second sweep stderr: Run pk=4: skipped (TBD window)
Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)
Run pk=39: skipped (TBD window)
Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)
Run pk=68: skipped (not approved)
Run pk=73: skipped (not approved)



## Proving D-04: a real sweep touches nothing outside its own namespace

The four runs seeded above already have their `CalendarEvent` rows from the real sweep two
cells up, and the second sweep above already proved that re-running `reconcile_campaign_runs`
against unchanged run state writes nothing. The cells below go one step further, against the
scratch copy of the developer database made in the setup cell above, not just this notebook's own four demo runs:
**every** `CalendarEvent` whose url falls outside the reconciler's own `RUN:` namespace --
classically-scheduled events, LCO/Gemini/ESO queue events, hand-entered conference/proposal-
deadline entries, everything -- is snapshotted, previewed, then swept for real, and the
before/after diff over that whole set must be empty. This is D-04's real-database half of
ROADMAP criterion 2's proof (the fixture-test half lives in plan 33-01's test suite).


Why the preview runs first: the reconciler is being inverted from an owner into an annotator,
so before trusting a full sweep against this developer's real data, the sweep is previewed in
Python first -- every url it could touch is computed from the reconciler's own key builders and
asserted to stay inside `RUN:`, and every pre-existing non-`RUN:` event already attributed to a
run is asserted to be outside that touchable set. Only once that preview holds does the real
sweep below become a confirmation of an already-checked claim, rather than the experiment
itself.


In [7]:
from datetime import timedelta

from django.core.exceptions import ObjectDoesNotExist
from django.core.management import call_command
from tom_calendar.models import CalendarEvent

from solsys_code.campaign_reconciler import (
    RUN_URL_NAMESPACE,
    owned_events,
    reconcile_run,
    run_container_url,
    run_night_url,
)
from solsys_code.models import CalendarEventMeta, CampaignRun
from solsys_code.solsys_code_observatory.models import Observatory


def snapshot_non_reconciler_events():
    """(pk -> url, title, description, start_time, end_time, attributed run_id) for every
    CalendarEvent whose url is NOT inside the reconciler's RUN: namespace."""
    snapshot = {}
    for event in CalendarEvent.objects.exclude(url__startswith=RUN_URL_NAMESPACE).select_related(
        'telescope_label_meta'
    ):
        try:
            run_id = event.telescope_label_meta.run_id
        except ObjectDoesNotExist:
            run_id = None
        snapshot[event.pk] = (
            event.url,
            event.title,
            event.description,
            event.start_time,
            event.end_time,
            run_id,
        )
    return snapshot


# --- Step 1: snapshot every non-RUN:-namespaced CalendarEvent, BEFORE any write below ---
pre_sweep_snapshot = snapshot_non_reconciler_events()
print(f'Step 1 -- snapshotted {len(pre_sweep_snapshot)} non-RUN:-namespaced CalendarEvent row(s) before any write.')

# --- Step 2: PRE-SWEEP DRY-RUN INSPECTION -- preview the sweep in Python before it runs ---
print()
print('=== PRE-SWEEP DRY-RUN INSPECTION ===')
print(
    f"{'run_pk':>7} {'created':>7} {'updated':>7} {'unchanged':>9} {'blocked':>7} "
    f"{'skipped_nights':>14}  skipped_reason"
)

touchable_urls: set[str] = set()
total_skipped_nights = 0

for run in CampaignRun.objects.all().select_related('site', 'campaign').order_by('pk'):
    try:
        result = reconcile_run(run, dry_run=True)
    except Exception as exc:  # noqa: BLE001 -- mirrors reconcile_campaign_runs.handle()'s own per-run catch
        print(f'{run.pk:>7}  reconcile_run(dry_run=True) raised {exc!r} -- skipping')
        continue

    print(
        f'{run.pk:>7} {result.created:>7} {result.updated:>7} {result.unchanged:>9} '
        f'{result.blocked:>7} {result.skipped_nights:>14}  {result.skipped_reason or ""}'
    )
    total_skipped_nights += result.skipped_nights

    if result.skipped_reason is not None:
        # Stage-0 guard fired (_skip_reason()) -- this run projects nothing, so it adds no
        # url to the touchable set.
        continue

    if run.telescope_class or (run.site is not None and run.site.observations_type == Observatory.SATELLITE_OBSTYPE):
        touchable_urls.add(run_container_url(run))
    elif run.window_start is not None and run.window_end is not None:
        n_nights = (run.window_end - run.window_start).days + 1
        touchable_urls |= {run_night_url(run, run.window_start + timedelta(days=i)) for i in range(n_nights)}
    # owned_events() is namespace-identity, so this can only ever add RUN:-prefixed urls --
    # included so the detach step's reach (T-29-19) is covered too, per the plan.
    touchable_urls |= {e.url for e in owned_events(run)}

print()
print(f'Summed skipped_nights across all runs: {total_skipped_nights}')

non_namespace_in_touchable = {u for u in touchable_urls if not u.startswith(RUN_URL_NAMESPACE)}
print(
    f'Touchable url set size: {len(touchable_urls)}; every member starts with '
    f'RUN_URL_NAMESPACE ({RUN_URL_NAMESPACE!r}): {not non_namespace_in_touchable}'
)
assert (
    not non_namespace_in_touchable
), f'Touchable set contains urls outside {RUN_URL_NAMESPACE!r}: {non_namespace_in_touchable}'

# The direct check on the pre-existing attributed non-RUN: row (CONTEXT.md's dev-DB baseline
# names exactly one such row): every CalendarEventMeta whose run is set and whose event's url
# is OUTSIDE the RUN: namespace must never be a url the sweep can touch.
foreign_attributed = (
    CalendarEventMeta.objects.filter(run__isnull=False)
    .exclude(event__url__startswith=RUN_URL_NAMESPACE)
    .select_related('event', 'run')
)
print()
print('CalendarEventMeta rows attributed to a run whose event url is OUTSIDE the RUN: namespace:')
foreign_urls = set()
for meta in foreign_attributed:
    print(f'  event pk={meta.event_id}  url={meta.event.url!r}  attributed to run pk={meta.run_id}')
    foreign_urls.add(meta.event.url)
if not foreign_urls:
    print('  (none)')

overlap = foreign_urls & touchable_urls
print(f'Any of those urls in the touchable set: {bool(overlap)}')
assert not overlap, f'The sweep must never be able to touch an attributed non-RUN: event: {overlap}'

# Cross-check: the management command's own --dry-run output is an AGGREGATE summary only --
# it sums created/updated/unchanged/blocked and prints no url and no per-run skipped_nights at
# all (33-REVIEWS.md Agreed Concern 4). The per-run table and the url-namespace assertions
# above come entirely from reconcile_run()'s returned ReconcileResult, never from parsing this
# command's stdout.
import io  # noqa: E402

stdout_buf = io.StringIO()
call_command('reconcile_campaign_runs', '--dry-run', stdout=stdout_buf)
print()
print('Aggregate summary only (reconcile_campaign_runs --dry-run stdout) -- the per-run detail')
print('and the url-namespace assertions above come from reconcile_run()s returned ReconcileResult,')
print('not from parsing this line:')
print(stdout_buf.getvalue())

Step 1 -- snapshotted 167 non-RUN:-namespaced CalendarEvent row(s) before any write.

=== PRE-SWEEP DRY-RUN INSPECTION ===
 run_pk created updated unchanged blocked skipped_nights  skipped_reason


      1       0       0        15       0              0  


      2       0       0         1       0              0  


      3       0       0         1       0              0  
      4       0       0         0       0              0  TBD window


      5       0       0         1       0              0  


...

Run pk=39: skipped (TBD window)


Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)


Run pk=68: skipped (not approved)



Aggregate summary only (reconcile_campaign_runs --dry-run stdout) -- the per-run detail
and the url-namespace assertions above come from reconcile_run()s returned ReconcileResult,
not from parsing this line:
Done (dry run). runs: 54, would_create: 0, would_update: 0, would_leave_unchanged: 82, skipped: 10, failed: 0, blocked: 0, skipped_nights: 0, would_detach: n/a (dry-run)



Run pk=73: skipped (not approved)


In [8]:
# --- Step 3: run the real full sweep, then compute the before/after diff ---
import io

from django.core.management import call_command

stdout_buf_real = io.StringIO()
call_command('reconcile_campaign_runs', stdout=stdout_buf_real)
print('Real sweep stdout:', stdout_buf_real.getvalue())

post_sweep_snapshot = snapshot_non_reconciler_events()

print()
print('=== POST-SWEEP DIFF ===')
compared_pks = set(pre_sweep_snapshot) | set(post_sweep_snapshot)
diff = {
    pk: (pre_sweep_snapshot.get(pk), post_sweep_snapshot.get(pk))
    for pk in compared_pks
    if pre_sweep_snapshot.get(pk) != post_sweep_snapshot.get(pk)
}
print(f'Compared {len(compared_pks)} non-RUN:-namespaced CalendarEvent row(s) before vs. after the real sweep.')
print(f'Differences found: {len(diff)}')
for pk, (before, after) in diff.items():
    print(f'  pk={pk}: before={before!r}  after={after!r}')
assert not diff, f'The real sweep must never change a non-RUN:-namespaced event: {diff}'
print('POST-SWEEP DIFF is empty -- the sweep changed nothing outside its own RUN: namespace.')

Run pk=4: skipped (TBD window)


Run pk=27: skipped (TBD window)
Run pk=28: skipped (TBD window)
Run pk=31: skipped (not approved)


Run pk=39: skipped (TBD window)


Run pk=42: skipped (TBD window)
Run pk=43: skipped (not approved)
Run pk=45: skipped (unresolved site)


Run pk=68: skipped (not approved)


Real sweep stdout: Done. runs: 54, created: 0, updated: 0, unchanged: 82, skipped: 10, failed: 0, blocked: 0, skipped_nights: 0, detached: 0


=== POST-SWEEP DIFF ===
Compared 167 non-RUN:-namespaced CalendarEvent row(s) before vs. after the real sweep.
Differences found: 0
POST-SWEEP DIFF is empty -- the sweep changed nothing outside its own RUN: namespace.


Run pk=73: skipped (not approved)


## Two new counters: `skipped_nights` and `detached`

`reconcile_campaign_runs` and `ReconcileResult` now report two more numbers alongside
`created`/`updated`/`unchanged`/`blocked` (33-REVIEW.md WR-01, WR-03):

- **`skipped_nights`** -- classical nights the sweep touched nothing for, because a
  non-`RUN:` event is already attributed to the run for that night (D-01, ANNOT-01).
  A non-zero `skipped_nights` alongside `created: 0, updated: 0` means "this run's
  nights are covered elsewhere", not "nothing changed".
- **`detached`** -- companion rows the sweep released back into Phase 28's attribution
  queue because the night they cover became attributed through a non-`RUN:` event
  *after* this reconciler had already minted its own event for it (CR-03). The
  `CalendarEvent` row itself survives -- detached, never deleted -- and its
  confirmation stamps (`confirmed_by`/`confirmed_at`) are cleared with the link.
  `--dry-run` always reports `would_detach: n/a (dry-run)`, because the detach step
  is itself a write and does not run in a dry run.

The skip-rule demo below shows both counters moving together in the
reconcile-then-attribute ordering.


## The skip rule: an attributed night gets no reconciler event

**This notebook runs against a throwaway copy of the developer database, made in the
setup cell above** -- every row it creates or removes dies with that copy, never
touching `src/fomo_db.sqlite3` itself. Every removal below is scoped to a row this SAME
cell created earlier in this SAME notebook run, by the primary key captured at creation
time -- never a blanket or unconditional delete (IN-04, 33-05 P2).

`classical_run`'s middle night already has its own `RUN:{pk}:{date}` event from this
notebook's earlier real sweep (the cell above). The night an event's start time
belongs to is anchored at local noon, matching `sun_event()`'s own convention
(`telescope_runs._local_noon_utc()`): an event starting after local midnight belongs
to the PREVIOUS date's observing night, not the date its own naive local `.date()`
would name (CR-02).

Seeding a blank-url stand-in event for that same night and attributing it directly
via a `CalendarEventMeta` row -- standing in for a night whose calendar entry already
comes from another writer (a classical-schedule loader, a hand entry, or Phase 34's
observation projector) -- and reconciling `classical_run` again shows the skip rule
(D-01, ANNOT-01) fire unconditionally on the attribution alone (CR-03, 33-REVIEW.md):
the attributed event's url is left unchanged, `ReconcileResult.skipped_nights` counts
the night, and the reconciler's now-superseded `RUN:{pk}:{date}` event is DETACHED
(never deleted) back into Phase 28's attribution queue -- `ReconcileResult.detached`
counts it, and its confirmation stamps are cleared with the link. This is the same
handoff rule Phase 35's allocation layer will reuse verbatim.


In [9]:
from datetime import timedelta
from datetime import timezone as dt_timezone

from solsys_code.campaign_attribution import orphan_calendar_events
from solsys_code.campaign_reconciler import run_night_url
from solsys_code.models import CalendarEventMeta
from solsys_code.telescope_runs import sun_event

# The run's own middle night -- already has its own RUN:-keyed event from this
# notebook's earlier real sweep (the cell above). With the skip now unconditional
# (CR-03, 33-REVIEW.md), the demo no longer needs to remove that event to make room
# for the attributed stand-in below -- the fix's own detach step reclaims it instead.
skip_demo_night = classical_run.window_start + timedelta(days=1)
existing_run_keyed_url = run_night_url(classical_run, skip_demo_night)
run_keyed_event = CalendarEvent.objects.get(url=existing_run_keyed_url)
run_keyed_pk = run_keyed_event.pk
print(
    f'classical_run pk={classical_run.pk} already owns {existing_run_keyed_url!r} '
    f'(pk={run_keyed_pk}) for the demo night.'
)

sunset, sunrise = sun_event(ground_site, skip_demo_night, kind='sun')
blank_url_event = CalendarEvent.objects.create(
    title='Hand-entered classical night (skip-rule demo)',
    description='A blank-url calendar entry standing in for a load_telescope_runs-created night.',
    start_time=sunset.to_datetime(timezone=dt_timezone.utc).replace(microsecond=0),
    end_time=sunrise.to_datetime(timezone=dt_timezone.utc).replace(microsecond=0),
    telescope='RDGS',
    instrument='EFOSC2',
)
blank_url_event_pk = blank_url_event.pk
CalendarEventMeta.objects.create(event=blank_url_event, run=classical_run)
print(
    f'Seeded blank-url event pk={blank_url_event_pk} url={blank_url_event.url!r} on night '
    f'{skip_demo_night}, attributed to classical_run pk={classical_run.pk}'
)

blank_url_before = blank_url_event.url
skip_result = reconcile_run(classical_run)
blank_url_event.refresh_from_db()
run_keyed_event.refresh_from_db()

print()
print(f'Reconcile result for classical_run pk={classical_run.pk}: {skip_result}')
print(f'(a) Attributed event url before reconcile: {blank_url_before!r}')
print(f'    Attributed event url after reconcile:  {blank_url_event.url!r}')
assert blank_url_event.url == blank_url_before, 'the skip rule must never re-key an attributed event'

print()
print(f'(b) skip_result.skipped_nights: {skip_result.skipped_nights}, ' f'skip_result.detached: {skip_result.detached}')
assert skip_result.skipped_nights == 1, f'expected exactly 1 skipped night, got {skip_result.skipped_nights}'
assert skip_result.detached == 1, f'expected exactly 1 detached event, got {skip_result.detached}'

print()
superseded_meta = CalendarEventMeta.objects.get(event_id=run_keyed_pk)
print(
    f'(c) The superseded RUN:-keyed event pk={run_keyed_pk} url={run_keyed_event.url!r} still '
    f'exists as a CalendarEvent: {CalendarEvent.objects.filter(pk=run_keyed_pk).exists()}'
)
print(f"    ...but its companion row's run is now: {superseded_meta.run_id!r} (detached, not deleted)")
assert CalendarEvent.objects.filter(pk=run_keyed_pk).exists(), 'the superseded event must survive -- never deleted'
assert superseded_meta.run_id is None, 'the superseded event must be detached (run cleared)'
assert run_keyed_pk in set(
    orphan_calendar_events().values_list('pk', flat=True)
), "the detached event must be back in Phase 28's attribution queue"

print()
night_attributions = CalendarEventMeta.objects.filter(
    event_id__in=[run_keyed_pk, blank_url_event_pk], run=classical_run
)
print(
    f'(d) entries attributed to classical_run for the {skip_demo_night} night: '
    f'{night_attributions.count()} (event pk(s): {list(night_attributions.values_list("event_id", flat=True))})'
)
assert night_attributions.count() == 1, 'exactly one entry for that night must be attributed to the run'

# Demo-scoped cleanup: remove ONLY the blank-url stand-in event this cell created, by
# the primary key captured at creation time above -- never the superseded RUN:-keyed
# event, which is left detached on the calendar for a human to re-confirm or discard
# via Phase 28's attribution queue (IN-04, 33-05 P2).
assert blank_url_event_pk, 'blank_url_event_pk must be captured before cleanup'
CalendarEvent.objects.filter(pk=blank_url_event_pk).delete()
print()
print(f'Demo-scoped cleanup: removed blank-url stand-in event pk={blank_url_event_pk} (created earlier in this cell).')

classical_run pk=59 already owns 'RUN:59:2026-09-02' (pk=335) for the demo night.


Seeded blank-url event pk=348 url='' on night 2026-09-02, attributed to classical_run pk=59


Reconcile detached 1 stale/superseded event(s) from run pk=59; confirmation stamps cleared.



Reconcile result for classical_run pk=59: ReconcileResult(created=0, updated=0, unchanged=2, blocked=0, skipped_nights=1, detached=1, skipped_reason=None)
(a) Attributed event url before reconcile: ''
    Attributed event url after reconcile:  ''

(b) skip_result.skipped_nights: 1, skip_result.detached: 1

(c) The superseded RUN:-keyed event pk=335 url='RUN:59:2026-09-02' still exists as a CalendarEvent: True
    ...but its companion row's run is now: None (detached, not deleted)

(d) entries attributed to classical_run for the 2026-09-02 night: 1 (event pk(s): [348])

Demo-scoped cleanup: removed blank-url stand-in event pk=348 (created earlier in this cell).


## Summary

`reconcile_campaign_runs` is for sweeps and backfills, not routine use: `approve()`,
`_resolve_site()`, `mark_cancelled` and `mark_weather_failure` in `campaign_views.py` each
call `campaign_reconciler.reconcile_run()` directly, so a single run's calendar events are
reconciled immediately the moment a staff member takes one of those actions on it -- no
command run is needed for that common case. The command demonstrated above exists for the
less common cases: reconciling every run at once (for example, after a bulk site repair),
or catching up a run whose state changed outside those four staff actions.

See `docs/runbooks/telescope_runs_calendar.rst`, "How do I get every campaign run onto the
calendar?", for the operator-facing version of everything demonstrated in this notebook.

## Scratch database teardown

Removes the scratch copy created in the setup cell above. The developer database
was never opened for writing by this notebook run.

In [10]:
import shutil

shutil.rmtree(scratch_db_dir, ignore_errors=True)
print(f'Removed scratch database directory: {scratch_db_dir}')
print('The developer database (src/fomo_db.sqlite3) was never opened for writing.')

Removed scratch database directory: /tmp/fomo-notebook-db-4d2o_ad7
The developer database (src/fomo_db.sqlite3) was never opened for writing.
